# NBA Data Scraping — Exploration

Exploration progressive des données disponibles via `nba_api`.
On explore chaque source avant de décider quoi garder.

**Saisons ciblées :** 2022-23, 2023-24, 2024-25  
**Cibles ML :** Victoire/Défaite (classification) + Point Diff (régression)

In [17]:
# pip install nba_api pandas pyarrow

import time
import os
import pandas as pd
import numpy as np

from nba_api.stats.endpoints import (
    leaguegamelog,
    boxscoretraditionalv3,
    boxscoreadvancedv3,
    boxscoresummaryv3,
    leaguedashteamstats,
    playergamelog,
    commonplayerinfo,
    leaguegamefinder,
    teamgamelog,
    commonteamroster,
    leaguedashplayerstats,
)
from nba_api.stats.static import teams, players

os.makedirs("data", exist_ok=True)

# Saisons au format nba_api
SEASONS = ["2022-23", "2023-24", "2024-25"]

# Sleep entre chaque appel pour éviter le rate limit NBA
SLEEP = 0.6

print("Imports OK")

Imports OK


## 1. Équipes NBA (données statiques)
Pas d'appel API — données embarquées dans nba_api.

In [18]:
nba_teams = teams.get_teams()
df_teams = pd.DataFrame(nba_teams)
print(f"Nombre d'équipes : {len(df_teams)}")
print(f"Colonnes : {df_teams.columns.tolist()}")
df_teams.head()

Nombre d'équipes : 30
Colonnes : ['id', 'full_name', 'abbreviation', 'nickname', 'city', 'state', 'year_founded']


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966


In [ ]:
# Sauvegarder les équipes
df_teams.to_csv("../../data/nba/nba_teams.csv", index=False)
print("Sauvegardé : ../../data/nba/nba_teams.csv")

Sauvegardé : data/nba_teams.csv


## 2. Calendrier complet — LeagueGameLog

**Endpoint :** `LeagueGameLog`  
**Ce qu'on obtient :** tous les matchs d'une saison avec scores, stats basiques  

Note : nba_api retourne une ligne PAR ÉQUIPE par match (format long).  
Il faudra pivoter pour avoir 1 ligne par match (home vs away).

In [20]:
# Exploration sur une saison d'abord
gamelog = leaguegamelog.LeagueGameLog(
    season="2024-25",
    season_type_all_star="Regular Season",
    direction="ASC",
)
time.sleep(SLEEP)

df_log = gamelog.get_data_frames()[0]
print(f"Shape : {df_log.shape}")
print(f"Colonnes : {df_log.columns.tolist()}")
df_log.head(4)

Shape : (2460, 29)
Colonnes : ['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE']


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,29,40,33,6,3,4,15,132,23,1
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,35,47,17,4,1,16,22,103,-7,1
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,31,46,22,7,8,7,22,110,7,1
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,29,34,20,2,3,12,12,109,-23,1


In [21]:
# GAME_ID identifie le match, MATCHUP indique home/away (@)
print("Exemple de matchup :")
print(df_log[["GAME_ID", "TEAM_ABBREVIATION", "MATCHUP", "WL", "PTS"]].head(6))
print()
print("Nombre de matchs uniques :", df_log["GAME_ID"].nunique())

Exemple de matchup :
      GAME_ID TEAM_ABBREVIATION      MATCHUP WL  PTS
0  0022400061               BOS  BOS vs. NYK  W  132
1  0022400062               MIN    MIN @ LAL  L  103
2  0022400062               LAL  LAL vs. MIN  W  110
3  0022400061               NYK    NYK @ BOS  L  109
4  0022400072               GSW    GSW @ POR  W  140
5  0022400068               HOU  HOU vs. CHA  L  105

Nombre de matchs uniques : 1230


In [22]:
# Pivot : transformer en 1 ligne par match (home vs away)
def gamelog_to_matches(df):
    """
    Transforme le format nba_api (1 ligne/équipe) en 1 ligne/match.
    MATCHUP format : 'BOS vs. MIA' (home) ou 'MIA @ BOS' (away)
    """
    home = df[df["MATCHUP"].str.contains(r"vs\.")].copy()
    away = df[df["MATCHUP"].str.contains(r"@")].copy()

    home = home.add_prefix("home_")
    away = away.add_prefix("away_")

    home = home.rename(columns={"home_GAME_ID": "GAME_ID", "home_GAME_DATE": "date"})
    away = away.rename(columns={"away_GAME_ID": "GAME_ID"})

    merged = home.merge(away, on="GAME_ID")

    # Cible ML
    merged["home_win"] = (merged["home_WL"] == "W").astype(int)
    merged["point_diff"] = merged["home_PTS"] - merged["away_PTS"]

    return merged

df_matches_sample = gamelog_to_matches(df_log)
print(f"Matchs 2024-25 : {len(df_matches_sample)}")
print(f"Colonnes : {len(df_matches_sample.columns)}")
df_matches_sample[["GAME_ID", "date", "home_TEAM_ABBREVIATION", "away_TEAM_ABBREVIATION",
                    "home_PTS", "away_PTS", "home_win", "point_diff"]].head()

Matchs 2024-25 : 1225
Colonnes : 59


,GAME_ID,date,home_TEAM_ABBREVIATION,away_TEAM_ABBREVIATION,home_PTS,away_PTS,home_win,point_diff
0,0022400061,2024-10-22,BOS,NYK,132,109,1,23
1,0022400062,2024-10-22,LAL,MIN,110,103,1,7
2,0022400068,2024-10-23,HOU,CHA,105,110,0,-5
3,0022400071,2024-10-23,LAC,PHX,113,116,0,-3
4,0022400072,2024-10-23,POR,GSW,104,140,0,-36


In [23]:
# Scraper les 3 saisons + playoffs
all_games = []

for season in SEASONS:
    for season_type in ["Regular Season", "Playoffs"]:
        gl = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star=season_type,
            direction="ASC",
        )
        time.sleep(SLEEP)
        df = gl.get_data_frames()[0]
        df["season"] = season
        df["season_type"] = season_type
        all_games.append(df)
        print(f"  {season} {season_type} : {df['GAME_ID'].nunique()} matchs")

df_all_raw = pd.concat(all_games, ignore_index=True)
print(f"\nTotal lignes brutes : {len(df_all_raw)}")
print(f"Total matchs uniques : {df_all_raw['GAME_ID'].nunique()}")

  2022-23 Regular Season : 1230 matchs
  2022-23 Playoffs : 84 matchs
  2023-24 Regular Season : 1230 matchs
  2023-24 Playoffs : 82 matchs
  2024-25 Regular Season : 1230 matchs
  2024-25 Playoffs : 84 matchs

Total lignes brutes : 7880
Total matchs uniques : 3940


In [24]:
# Pivot en 1 ligne par match
df_matches = gamelog_to_matches(df_all_raw)
print(f"DataFrame final : {df_matches.shape}")
print(f"Colonnes disponibles :\n{df_matches.columns.tolist()}")
df_matches.head(3)

DataFrame final : (3935, 63)
Colonnes disponibles :
['home_SEASON_ID', 'home_TEAM_ID', 'home_TEAM_ABBREVIATION', 'home_TEAM_NAME', 'GAME_ID', 'date', 'home_MATCHUP', 'home_WL', 'home_MIN', 'home_FGM', 'home_FGA', 'home_FG_PCT', 'home_FG3M', 'home_FG3A', 'home_FG3_PCT', 'home_FTM', 'home_FTA', 'home_FT_PCT', 'home_OREB', 'home_DREB', 'home_REB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF', 'home_PTS', 'home_PLUS_MINUS', 'home_VIDEO_AVAILABLE', 'home_season', 'home_season_type', 'away_SEASON_ID', 'away_TEAM_ID', 'away_TEAM_ABBREVIATION', 'away_TEAM_NAME', 'away_GAME_DATE', 'away_MATCHUP', 'away_WL', 'away_MIN', 'away_FGM', 'away_FGA', 'away_FG_PCT', 'away_FG3M', 'away_FG3A', 'away_FG3_PCT', 'away_FTM', 'away_FTA', 'away_FT_PCT', 'away_OREB', 'away_DREB', 'away_REB', 'away_AST', 'away_STL', 'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_PLUS_MINUS', 'away_VIDEO_AVAILABLE', 'away_season', 'away_season_type', 'home_win', 'point_diff']


,home_SEASON_ID,home_TEAM_ID,home_TEAM_ABBREVIATION,home_TEAM_NAME,GAME_ID,date,home_MATCHUP,home_WL,home_MIN,home_FGM,...,away_BLK,away_TOV,away_PF,away_PTS,away_PLUS_MINUS,away_VIDEO_AVAILABLE,away_season,away_season_type,home_win,point_diff
0,22022,1610612738,BOS,Boston Celtics,0022200001,2022-10-18,BOS vs. PHI,W,240,46,...,3,14,25,117,-9,1,2022-23,Regular Season,1,9
1,22022,1610612744,GSW,Golden State Warriors,0022200002,2022-10-18,GSW vs. LAL,W,240,45,...,4,22,18,109,-14,1,2022-23,Regular Season,1,14
2,22022,1610612737,ATL,Atlanta Hawks,0022200005,2022-10-19,ATL vs. HOU,W,240,45,...,3,16,20,107,-10,1,2022-23,Regular Season,1,10


In [25]:
# Stats rapides
print("Répartition par saison :")
print(df_matches.groupby(["home_season", "home_season_type"])["GAME_ID"].count())
print()
print(f"Taux victoire domicile : {df_matches['home_win'].mean():.1%}")
print(f"Point diff moyen : {df_matches['point_diff'].mean():.1f}")

Répartition par saison :
home_season  home_season_type
2022-23      Playoffs              84
             Regular Season      1230
2023-24      Playoffs              82
             Regular Season      1230
2024-25      Playoffs              84
             Regular Season      1225
Name: GAME_ID, dtype: int64

Taux victoire domicile : 55.8%
Point diff moyen : 2.3


In [ ]:
df_matches.to_csv("../../data/nba/nba_games.csv", index=False)
print("Sauvegardé : ../../data/nba/nba_games.csv")

Sauvegardé : data/nba_games.csv


## 3. Stats avancées équipe — BoxScoreAdvancedV3

**Endpoint :** `BoxScoreAdvancedV3`  
**On obtient :** Net Rating, Pace, eFG%, TS% par match  

1 appel par match → pour 3 saisons (~3 700 matchs) = ~37 min avec sleep 0.6s

In [28]:
# Exploration sur 1 match d'abord pour voir
sample_game_id = df_matches["GAME_ID"].iloc[0]
print(f"Game ID exemple : {sample_game_id}")

adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=sample_game_id)
time.sleep(SLEEP)

# Deux DataFrames : team stats et player stats
df_adv_team   = adv.team_stats.get_data_frame()
df_adv_player = adv.player_stats.get_data_frame()

print(f"Team stats colonnes : {df_adv_team.columns.tolist()}")
print()
df_adv_team

Game ID exemple : 0022200001
Team stats colonnes : ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'estimatedTeamTurnoverPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']



,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,minutes,estimatedOffensiveRating,offensiveRating,estimatedDefensiveRating,...,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,0022200001,1610612738,Boston,Celtics,BOS,celtics,240:00,649.5,129.9,596.9,...,11.3,0.634,0.668,1.0,0.981,19.5,97.5,81.25,97.0,0.566
1,0022200001,1610612755,Philadelphia,76ers,PHI,sixers,240:00,596.9,119.4,649.5,...,14.3,0.581,0.634,1.0,0.962,19.5,97.5,81.25,98.0,0.434


In [31]:
ADV_COLS = [
    "gameId", "teamId", "teamTricode",
    "estimatedOffensiveRating", "estimatedDefensiveRating", "estimatedNetRating",
    "estimatedPace", "assistRatio",
    "offensiveReboundPercentage", "defensiveReboundPercentage", "reboundPercentage",
    "estimatedTeamTurnoverPercentage",
    "effectiveFieldGoalPercentage", "trueShootingPercentage", "PIE"
]

print("Colonnes disponibles vs souhaitées :")
for c in ADV_COLS:
    status = "✓" if c in df_adv_team.columns else "✗ ABSENT"
    print(f"  {status}  {c}")

df_adv_team[ADV_COLS]

Colonnes disponibles vs souhaitées :
  ✓  gameId
  ✓  teamId
  ✓  teamTricode
  ✓  estimatedOffensiveRating
  ✓  estimatedDefensiveRating
  ✓  estimatedNetRating
  ✓  estimatedPace
  ✓  assistRatio
  ✓  offensiveReboundPercentage
  ✓  defensiveReboundPercentage
  ✓  reboundPercentage
  ✓  estimatedTeamTurnoverPercentage
  ✓  effectiveFieldGoalPercentage
  ✓  trueShootingPercentage
  ✓  PIE


,gameId,teamId,teamTricode,estimatedOffensiveRating,estimatedDefensiveRating,estimatedNetRating,estimatedPace,assistRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,estimatedTeamTurnoverPercentage,effectiveFieldGoalPercentage,trueShootingPercentage,PIE
0,0022200001,1610612738,BOS,649.5,596.9,52.5,19.5,18.5,0.256,0.810,0.543,11.340,0.634,0.668,0.566
1,0022200001,1610612755,PHI,596.9,649.5,-52.5,19.5,13.1,0.190,0.744,0.457,14.286,0.581,0.634,0.434


In [32]:
# Scraper les stats avancées pour TOUS les matchs
# Pour tester, commencer par les 10 premiers matchs

game_ids = df_matches["GAME_ID"].unique()
print(f"Total matchs à scraper : {len(game_ids)}")
print("Estimation temps : ~{:.0f} min".format(len(game_ids) * SLEEP / 60))
print()
print("→ Lancer la cellule suivante pour scraper (commentez la ligne [:10] pour tout scraper)")

Total matchs à scraper : 3935
Estimation temps : ~39 min

→ Lancer la cellule suivante pour scraper (commentez la ligne [:10] pour tout scraper)


In [35]:
# Scraping stats avancées
# Retirez [:10] pour scraper tous les matchs
adv_rows = []
ids_to_scrape = game_ids

for i, gid in enumerate(ids_to_scrape):
    try:
        adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=gid)
        df_t = adv.team_stats.get_data_frame()
        adv_rows.append(df_t)
    except Exception as e:
        print(f"  Erreur {gid} : {e}")
    time.sleep(SLEEP)
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(ids_to_scrape)} matchs traités")

df_adv_all = pd.concat(adv_rows, ignore_index=True)
print(f"\nShape : {df_adv_all.shape}")
df_adv_all.head()

  50/3935 matchs traités
  100/3935 matchs traités
  150/3935 matchs traités
  200/3935 matchs traités
  250/3935 matchs traités
  300/3935 matchs traités
  350/3935 matchs traités
  400/3935 matchs traités
  450/3935 matchs traités
  500/3935 matchs traités
  550/3935 matchs traités
  600/3935 matchs traités
  650/3935 matchs traités
  700/3935 matchs traités
  750/3935 matchs traités
  800/3935 matchs traités
  850/3935 matchs traités
  900/3935 matchs traités
  950/3935 matchs traités
  1000/3935 matchs traités
  1050/3935 matchs traités
  1100/3935 matchs traités
  1150/3935 matchs traités
  1200/3935 matchs traités
  1250/3935 matchs traités
  1300/3935 matchs traités
  1350/3935 matchs traités
  1400/3935 matchs traités
  1450/3935 matchs traités
  1500/3935 matchs traités
  1550/3935 matchs traités
  1600/3935 matchs traités
  1650/3935 matchs traités
  1700/3935 matchs traités
  1750/3935 matchs traités
  1800/3935 matchs traités
  1850/3935 matchs traités
  1900/3935 matchs tr

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,minutes,estimatedOffensiveRating,offensiveRating,estimatedDefensiveRating,...,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,0022200001,1610612738,Boston,Celtics,BOS,celtics,240:00,649.5,129.9,596.9,...,11.3,0.634,0.668,1.0,0.981,19.5,97.5,81.25,97.0,0.566
1,0022200001,1610612755,Philadelphia,76ers,PHI,sixers,240:00,596.9,119.4,649.5,...,14.3,0.581,0.634,1.0,0.962,19.5,97.5,81.25,98.0,0.434
2,0022200002,1610612744,Golden State,Warriors,GSW,warriors,240:00,534.8,107.0,486.6,...,15.7,0.535,0.564,1.0,0.962,22.7,113.5,94.58,115.0,0.548
3,0022200002,1610612747,Los Angeles,Lakers,LAL,lakers,240:00,486.6,97.3,534.8,...,19.6,0.479,0.519,1.0,0.992,22.7,113.5,94.58,112.0,0.452
4,0022200005,1610612737,Atlanta,Hawks,ATL,hawks,240:00,551.9,110.4,504.7,...,8.5,0.539,0.582,1.0,0.982,21.2,106.0,88.33,106.0,0.572


In [ ]:
df_adv_all.to_csv("../../data/nba/nba_boxscore_advanced.csv", index=False)
print("Sauvegardé : ../../data/nba/nba_boxscore_advanced.csv")

Sauvegardé : data/nba_boxscore_advanced.csv


## 4. Stats traditionnelles équipe — BoxScoreTraditionalV3

**Endpoint :** `BoxScoreTraditionalV2`  
**On obtient :** Points, rebounds, assists, FG%, 3P%, turnovers  


In [37]:
# Explorer sur un match
trad = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=sample_game_id)
time.sleep(SLEEP)

df_trad_team = trad.team_stats.get_data_frame()
print(f"Colonnes : {df_trad_team.columns.tolist()}")
df_trad_team

Colonnes : ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted', 'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers', 'foulsPersonal', 'points', 'plusMinusPoints']


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,minutes,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,...,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints
0,0022200001,1610612738,Boston,Celtics,BOS,celtics,240:00,46,82,0.561,...,6,30,36,24,8,3,10,24,126,9.0
1,0022200001,1610612755,Philadelphia,76ers,PHI,sixers,240:00,40,80,0.500,...,4,27,31,16,8,3,14,25,117,-9.0


## 5. Features fatigue — Back-to-back & jours de repos

Calculées depuis `nba_games.csv` — pas d'appel API supplémentaire.  
C'est l'une des features les plus prédictives en NBA.

**Équivalent football :** jours de repos + distance déplacement

In [ ]:
df = pd.read_csv("../../data/nba/nba_games.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

def compute_rest_days(df):
    """
    Calcule les jours de repos pour chaque équipe avant chaque match.
    Back-to-back = 0 jours de repos (match la veille).
    """
    records = []
    
    # Construire l'historique par équipe
    for _, row in df.iterrows():
        records.append({"GAME_ID": row["GAME_ID"], "date": row["date"],
                         "team": row["home_TEAM_ABBREVIATION"], "side": "home"})
        records.append({"GAME_ID": row["GAME_ID"], "date": row["date"],
                         "team": row["away_TEAM_ABBREVIATION"], "side": "away"})
    
    team_history = pd.DataFrame(records).sort_values(["team", "date"])
    team_history["prev_date"] = team_history.groupby("team")["date"].shift(1)
    team_history["rest_days"] = (team_history["date"] - team_history["prev_date"]).dt.days - 1
    team_history["rest_days"] = team_history["rest_days"].fillna(7).clip(0, 14)
    team_history["is_b2b"] = (team_history["rest_days"] == 0).astype(int)
    
    # Rejoindre sur le DataFrame principal
    home_rest = team_history[team_history["side"] == "home"][["GAME_ID", "rest_days", "is_b2b"]]
    home_rest.columns = ["GAME_ID", "home_rest_days", "home_is_b2b"]
    away_rest = team_history[team_history["side"] == "away"][["GAME_ID", "rest_days", "is_b2b"]]
    away_rest.columns = ["GAME_ID", "away_rest_days", "away_is_b2b"]
    
    df = df.merge(home_rest, on="GAME_ID").merge(away_rest, on="GAME_ID")
    df["rest_advantage"] = df["home_rest_days"] - df["away_rest_days"]
    return df

df = compute_rest_days(df)

print("Taux de back-to-back :")
print(f"  Domicile B2B : {df['home_is_b2b'].mean():.1%}")
print(f"  Extérieur B2B : {df['away_is_b2b'].mean():.1%}")
print()
print("Impact B2B sur le taux de victoire domicile :")
print(df.groupby(["home_is_b2b", "away_is_b2b"])["home_win"].mean().round(3))

Taux de back-to-back :
  Domicile B2B : 14.9%
  Extérieur B2B : 17.4%

Impact B2B sur le taux de victoire domicile :
home_is_b2b  away_is_b2b
0            0              0.557
             1              0.628
1            0              0.473
             1              0.557
Name: home_win, dtype: float64


## 6. Stats joueurs — PlayerGameLog

**Endpoint :** `PlayerGameLog`  
**Ce qu'on obtient :** box score par match pour chaque joueur  

1 appel par joueur par saison (~500 joueurs × 3 saisons = ~1 500 appels = ~15 min)

In [39]:
# Explorer sur un joueur (LeBron James ID = 2544)
pgl = playergamelog.PlayerGameLog(player_id="2544", season="2024-25")
time.sleep(SLEEP)

df_pgl = pgl.get_data_frames()[0]
print(f"Colonnes : {df_pgl.columns.tolist()}")
print(f"Matchs : {len(df_pgl)}")
df_pgl.head(3)

Colonnes : ['SEASON_ID', 'Player_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE']
Matchs : 70


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,2544,0022401185,"Apr 11, 2025",LAL vs. HOU,W,22,6,11,0.545,...,4,4,8,1,0,1,1,14,5,1
1,22024,2544,0022401159,"Apr 09, 2025",LAL @ DAL,W,36,11,20,0.550,...,5,7,3,1,0,2,2,27,18,1
2,22024,2544,0022401153,"Apr 08, 2025",LAL @ OKC,L,35,8,19,0.421,...,7,7,3,1,0,6,1,28,-23,1


In [40]:
# Récupérer les joueurs actifs
all_players = players.get_active_players()
df_players = pd.DataFrame(all_players)
print(f"Joueurs actifs : {len(df_players)}")
df_players.head(3)

Joueurs actifs : 530


,id,full_name,first_name,last_name,is_active
0,1630173,Precious Achiuwa,Precious,Achiuwa,True
1,203500,Steven Adams,Steven,Adams,True
2,1628389,Bam Adebayo,Bam,Adebayo,True


In [42]:
# Scraper les stats joueurs pour 3 saisons
# Retirez [:5] pour tout scraper
player_stats_rows = []

for i, player in enumerate(all_players):
    for season in SEASONS:
        try:
            pgl = playergamelog.PlayerGameLog(
                player_id=str(player["id"]),
                season=season,
            )
            df_p = pgl.get_data_frames()[0]
            df_p["player_id"]   = player["id"]
            df_p["player_name"] = player["full_name"]
            df_p["season"]      = season
            player_stats_rows.append(df_p)
        except Exception as e:
            pass  # Joueur sans données pour cette saison
        time.sleep(SLEEP)
    
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(all_players)} joueurs")

df_player_stats = pd.concat(player_stats_rows, ignore_index=True)
print(f"Shape : {df_player_stats.shape}")
df_player_stats.head(3)

  50/530 joueurs
  100/530 joueurs
  150/530 joueurs
  200/530 joueurs
  250/530 joueurs
  300/530 joueurs
  350/530 joueurs
  400/530 joueurs
  450/530 joueurs
  500/530 joueurs
Shape : (62149, 30)


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,player_id,player_name,season
0,22022,1630173,0022201221,"Apr 09, 2023",TOR vs. MIL,W,28,6,11,0.545,...,2,0,2,2,14,9,1,1630173,Precious Achiuwa,2022-23
1,22022,1630173,0022201206,"Apr 07, 2023",TOR @ BOS,L,24,6,10,0.6,...,1,0,0,3,16,-9,1,1630173,Precious Achiuwa,2022-23
2,22022,1630173,0022201192,"Apr 05, 2023",TOR @ BOS,L,20,7,11,0.636,...,0,1,1,2,16,-4,1,1630173,Precious Achiuwa,2022-23


In [ ]:
df_player_stats.to_csv("../../data/nba/nba_player_stats.csv", index=False)
print("Sauvegardé : ../../data/nba/nba_player_stats.csv")

Sauvegardé : data/nba_player_stats.csv


## 7. Injury Report — Joueurs indisponibles

**Endpoint :** `CommonPlayerInfo`  
**Ce qu'on obtient :** statut de disponibilité d'un joueur  

L'injury report officiel NBA n'est pas historisé dans nba_api.  
On peut calculer un proxy : joueur absent d'un match = indisponible.

In [44]:
# Explorer CommonPlayerInfo
info = commonplayerinfo.CommonPlayerInfo(player_id="2544")
time.sleep(SLEEP)

df_info = info.get_data_frames()[0]
print(f"Colonnes : {df_info.columns.tolist()}")
df_info[["DISPLAY_FIRST_LAST", "TEAM_NAME", "POSITION", "HEIGHT",
          "WEIGHT", "COUNTRY", "DRAFT_YEAR", "DRAFT_ROUND"]]

Colonnes : ['PERSON_ID', 'FIRST_NAME', 'LAST_NAME', 'DISPLAY_FIRST_LAST', 'DISPLAY_LAST_COMMA_FIRST', 'DISPLAY_FI_LAST', 'PLAYER_SLUG', 'BIRTHDATE', 'SCHOOL', 'COUNTRY', 'LAST_AFFILIATION', 'HEIGHT', 'WEIGHT', 'SEASON_EXP', 'JERSEY', 'POSITION', 'ROSTERSTATUS', 'GAMES_PLAYED_CURRENT_SEASON_FLAG', 'TEAM_ID', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'TEAM_CODE', 'TEAM_CITY', 'PLAYERCODE', 'FROM_YEAR', 'TO_YEAR', 'DLEAGUE_FLAG', 'NBA_FLAG', 'GAMES_PLAYED_FLAG', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GREATEST_75_FLAG']


,DISPLAY_FIRST_LAST,TEAM_NAME,POSITION,HEIGHT,WEIGHT,COUNTRY,DRAFT_YEAR,DRAFT_ROUND
0,LeBron James,Lakers,Forward,6-9,250,USA,2003,1


In [45]:
# Proxy absences : calculer les matchs manqués par joueur
# Un joueur absent d'un match alors que son équipe jouait = indisponible

if "df_player_stats" in dir() and len(df_player_stats) > 0:
    # Pour chaque match, combien de joueurs de l'équipe ont joué ?
    players_per_game = (
        df_player_stats
        .groupby(["Game_ID", "player_id"])["MIN"]
        .sum()
        .reset_index()
    )
    print("Exemple stats joueurs par match :")
    print(players_per_game.head())
else:
    print("Scraper d'abord les stats joueurs (section 6)")

Exemple stats joueurs par match :
      Game_ID  player_id MIN
0  0022200001     201143  23
1  0022200001     201935  37
2  0022200001     202699  34
3  0022200001     203935  36
4  0022200001     203954  37


## 8. Stats agrégées équipe — LeagueDashTeamStats

**Endpoint :** `LeagueDashTeamStats`  
**Ce qu'on obtient :** stats cumulées par équipe sur N derniers matchs  

Paramètre clé : `last_n_games` pour la fenêtre glissante.

In [46]:
# Stats sur les 10 derniers matchs (forme récente)
dash = leaguedashteamstats.LeagueDashTeamStats(
    season="2024-25",
    last_n_games=10,
    measure_type_detailed_defense="Advanced",
)
time.sleep(SLEEP)

df_dash = dash.get_data_frames()[0]
print(f"Colonnes : {df_dash.columns.tolist()}")
print(f"Équipes : {len(df_dash)}")
df_dash[["TEAM_NAME", "W", "L", "W_PCT",
          "OFF_RATING", "DEF_RATING", "NET_RATING",
          "PACE", "EFG_PCT", "TS_PCT"]].head(8)

Colonnes : ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK']
Équipes : 30


,TEAM_NAME,W,L,W_PCT,OFF_RATING,DEF_RATING,NET_RATING,PACE,EFG_PCT,TS_PCT
0,Atlanta Hawks,5,5,0.5,120.4,117.0,3.4,101.80,0.585,0.614
1,Boston Celtics,8,2,0.8,120.4,108.5,12.0,93.58,0.556,0.584
2,Brooklyn Nets,3,7,0.3,103.9,117.1,-13.2,99.90,0.509,0.545
3,Charlotte Hornets,1,9,0.1,102.0,118.2,-16.2,98.10,0.495,0.530
4,Chicago Bulls,7,3,0.7,114.9,110.5,4.3,105.65,0.565,0.600
5,Cleveland Cavaliers,6,4,0.6,116.6,113.8,2.8,100.41,0.563,0.584
6,Dallas Mavericks,4,6,0.4,107.0,115.9,-9.0,100.55,0.541,0.573
7,Denver Nuggets,5,5,0.5,118.7,114.8,3.9,99.92,0.572,0.605


## 9. Récapitulatif — Données disponibles

Résumé de ce qu'on a collecté et ce qu'on garde pour le feature engineering.

In [ ]:
import os

print("=== FICHIERS GÉNÉRÉS ===")
for f in sorted(os.listdir("../../data/nba")):
    if f.startswith("nba_"):
        path = f"../../data/nba/{f}"
        size = os.path.getsize(path) / 1e6
        df_tmp = pd.read_csv(path)
        print(f"  {f:45s} {len(df_tmp):>8,} lignes · {size:.1f} MB")

print()
print("=== FEATURES DISPONIBLES POUR LE ML ===")
features = {
    "nba_games.csv":              "Résultats, scores, stats basiques — CIBLE : home_win + point_diff",
    "nba_boxscore_advanced.csv":  "Net Rating, Pace, eFG%, TS% par match",
    "nba_player_stats.csv":       "Box score joueurs (points, min, +/-...)",
    "nba_teams.csv":              "Métadonnées équipes",
    r"[calculé] rest_days":           "Jours de repos + back-to-back",
}
for f, desc in features.items():
    print(f"  {f:40s} → {desc}")

=== FICHIERS GÉNÉRÉS ===
  nba_boxscore_advanced.csv                        7,870 lignes · 1.5 MB
  nba_games.csv                                    3,935 lignes · 1.3 MB
  nba_player_stats.csv                            62,149 lignes · 8.5 MB
  nba_teams.csv                                       30 lignes · 0.0 MB

=== FEATURES DISPONIBLES POUR LE ML ===
  nba_games.csv                            → Résultats, scores, stats basiques — CIBLE : home_win + point_diff
  nba_boxscore_advanced.csv                → Net Rating, Pace, eFG%, TS% par match
  nba_player_stats.csv                     → Box score joueurs (points, min, +/-...)
  nba_teams.csv                            → Métadonnées équipes
  [calculé] rest_days                      → Jours de repos + back-to-back


In [ ]:
# ============================================================
# 10. Saison 2025-26 — Données pour la prédiction en production
# ============================================================

CURRENT_SEASON = "2025-26"

# ── 10.1 Matchs + résultats saison en cours ──────────────────
gl_current = leaguegamelog.LeagueGameLog(
    season=CURRENT_SEASON,
    season_type_all_star="Regular Season",
    direction="ASC",
)
time.sleep(SLEEP)

df_current_raw = gl_current.get_data_frames()[0]
df_current_raw["season"] = CURRENT_SEASON
df_current_raw["season_type"] = "Regular Season"

df_current = gamelog_to_matches(df_current_raw)
print(f"Matchs 2025-26 joués : {len(df_current)}")
df_current.to_csv("../../data/nba/nba_games_2526.csv", index=False)

# ── 10.2 Stats avancées équipes sur la saison en cours ────────
# Net Rating, Pace, eFG% — snapshot actuel de chaque équipe
dash_current = leaguedashteamstats.LeagueDashTeamStats(
    season=CURRENT_SEASON,
    season_type_all_star="Regular Season",
    measure_type_detailed_defense="Advanced",
    per_mode_detailed="PerGame",
)
time.sleep(SLEEP)

df_team_stats_current = dash_current.get_data_frames()[0]
print(f"Équipes avec stats 25/26 : {len(df_team_stats_current)}")
print(df_team_stats_current[["TEAM_NAME", "W", "L", "W_PCT",
                               "OFF_RATING", "DEF_RATING", "NET_RATING",
                               "PACE", "EFG_PCT"]].to_string())
df_team_stats_current.to_csv("../../data/nba/nba_team_stats_current.csv", index=False)

# ── 10.3 Stats équipes sur les 10 derniers matchs (forme récente) ──
dash_recent = leaguedashteamstats.LeagueDashTeamStats(
    season=CURRENT_SEASON,
    season_type_all_star="Regular Season",
    measure_type_detailed_defense="Advanced",
    per_mode_detailed="PerGame",
    last_n_games=10,
)
time.sleep(SLEEP)

df_team_recent = dash_recent.get_data_frames()[0]
df_team_recent = df_team_recent.add_prefix("recent_")
df_team_recent = df_team_recent.rename(columns={"recent_TEAM_ID": "TEAM_ID"})
print(f"\nForme récente (10 derniers matchs) : {len(df_team_recent)} équipes")
df_team_recent.to_csv("../../data/nba/nba_team_stats_recent.csv", index=False)

# ── 10.4 Roster actuel de chaque équipe ───────────────────────
from nba_api.stats.endpoints import commonteamroster

roster_rows = []
for team in nba_teams:
    try:
        r = commonteamroster.CommonTeamRoster(
            team_id=str(team["id"]),
            season=CURRENT_SEASON,
        )
        time.sleep(SLEEP)
        df_r = r.get_data_frames()[0]
        df_r["team_id"]           = team["id"]
        df_r["team_abbreviation"] = team["abbreviation"]
        roster_rows.append(df_r)
    except Exception as e:
        print(f"  Erreur roster {team['abbreviation']} : {e}")

df_rosters = pd.concat(roster_rows, ignore_index=True)
print(f"\nRoster 2025-26 : {len(df_rosters)} joueurs")
print(f"Colonnes : {df_rosters.columns.tolist()}")
df_rosters.to_csv("../../data/nba/nba_rosters_current.csv", index=False)

# ── 10.5 Stats joueurs saison en cours ────────────────────────
from nba_api.stats.endpoints import leaguedashplayerstats

player_stats_current = leaguedashplayerstats.LeagueDashPlayerStats(
    season=CURRENT_SEASON,
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame",
)
time.sleep(SLEEP)

df_player_current = player_stats_current.get_data_frames()[0]
print(f"\nStats joueurs 2025-26 : {len(df_player_current)} joueurs")
print(f"Colonnes : {df_player_current.columns.tolist()}")
df_player_current.to_csv("../../data/nba/nba_player_stats_current.csv", index=False)

# ── 10.6 Injury Report actuel ─────────────────────────────────

try:
    from nba_api.stats.endpoints import leagueinjuryfinder
    injury = leagueinjuryfinder.LeagueInjuryFinder()
    time.sleep(SLEEP)
    df_injury = injury.get_data_frames()[0]
    print(f"\nInjury report : {len(df_injury)} joueurs concernés")
    df_injury.to_csv("../../data/nba/nba_injury_report.csv", index=False)
except Exception as e:
    print(f"Injury report indisponible : {e}")
    print("→ Les indisponibilités seront saisies manuellement via le chat LLM")
    # Créer un fichier vide pour la cohérence
    pd.DataFrame(columns=["player_name", "team", "status", "reason"]).to_csv(
        "../../data/nba/nba_injury_report.csv", index=False
    )

Matchs 2025-26 joués : 1225
Équipes avec stats 25/26 : 30
                 TEAM_NAME   W   L  W_PCT  OFF_RATING  DEF_RATING  NET_RATING    PACE  EFG_PCT
0            Atlanta Hawks  46  36  0.561       115.0       112.9         2.2  102.50    0.554
1           Boston Celtics  56  26  0.683       120.0       111.7         8.3   95.58    0.553
2            Brooklyn Nets  20  62  0.244       108.2       118.2       -10.0   97.60    0.520
3        Charlotte Hornets  44  38  0.537       118.4       113.5         4.9   97.60    0.552
4            Chicago Bulls  31  51  0.378       112.1       117.4        -5.3  103.22    0.547
5      Cleveland Cavaliers  52  30  0.634       118.3       114.1         4.1  100.70    0.561
6         Dallas Mavericks  26  56  0.317       110.3       115.5        -5.2  102.63    0.527
7           Denver Nuggets  54  28  0.659       121.2       116.0         5.2   99.49    0.577
8          Detroit Pistons  60  22  0.732       117.3       108.9         8.4   99.88  

In [ ]:
for f in sorted(os.listdir("../../data/nba")):
    if f.startswith("nba_"):
        path = f"../../data/nba/{f}"
        df_tmp = pd.read_csv(path)
        print(f"{f:50s} {len(df_tmp):>8,} lignes · {df_tmp.shape[1]} colonnes")

nba_boxscore_advanced.csv                             7,870 lignes · 30 colonnes
nba_games.csv                                         3,935 lignes · 63 colonnes
nba_games_2526.csv                                    1,225 lignes · 63 colonnes
nba_injury_report.csv                                     0 lignes · 4 colonnes
nba_player_stats.csv                                 62,149 lignes · 30 colonnes
nba_player_stats_current.csv                            582 lignes · 67 colonnes
nba_rosters_current.csv                                 530 lignes · 18 colonnes
nba_team_stats_current.csv                               30 lignes · 46 colonnes
nba_team_stats_recent.csv                                30 lignes · 46 colonnes
nba_teams.csv                                            30 lignes · 7 colonnes


In [ ]:
# Vérification les colonnes de jointure
games = pd.read_csv("../../data/nba/nba_games.csv")
adv   = pd.read_csv("../../data/nba/nba_boxscore_advanced.csv")

print("=== nba_games colonnes clés ===")
print([c for c in games.columns if any(x in c.lower() 
      for x in ["id", "team", "date", "season"])])

print("\n=== nba_boxscore_advanced colonnes clés ===")
print([c for c in adv.columns if any(x in c.lower() 
      for x in ["id", "team", "date", "season"])])

=== nba_games colonnes clés ===
['home_SEASON_ID', 'home_TEAM_ID', 'home_TEAM_ABBREVIATION', 'home_TEAM_NAME', 'GAME_ID', 'date', 'home_VIDEO_AVAILABLE', 'home_season', 'home_season_type', 'away_SEASON_ID', 'away_TEAM_ID', 'away_TEAM_ABBREVIATION', 'away_TEAM_NAME', 'away_GAME_DATE', 'away_VIDEO_AVAILABLE', 'away_season', 'away_season_type']

=== nba_boxscore_advanced colonnes clés ===
['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'estimatedTeamTurnoverPercentage']
